# Session 1: Language Model Concept with Feedforward Neural Network

This notebook covers the first session only.

Goal: understand the concept of a simple language model before training a model in the next session.

Topics:

1. What is a Language Model?
2. Next-word Prediction
3. Tokenization
4. Vocabulary
5. Word IDs
6. N-gram Concept
7. Context Window
8. Context-next Word Pairs
9. Next-word Prediction as Classification
10. Basic Feedforward Neural Network Idea
11. Limitation of FNN Language Models

## 1. What is a Language Model?

A **Language Model** is a model that learns patterns from text.

One simple way to describe a language model is:

> A language model tries to predict what text is likely to come next.

For example, if we write:

```text
I like to eat
```

a language model may predict:

```text
rice
noodles
pizza
```

The model does not truly understand food like a human does. It learns from examples of text and uses patterns from those examples.

In this notebook, we will study a very simple language model: **predict the next word from previous words**.

## 2. Next-word Prediction

Next-word prediction means:

```text
previous words -> next word
```

### Human Example

If a human sees:

```text
I drink water when I am
```

The human may guess:

```text
thirsty
```

The human uses experience, meaning, and common sense.

### Language Model Example

A simple language model does not use common sense. It looks at patterns from training text.

If the model often saw:

```text
i like to eat rice
i like to eat noodles
i like to eat fruit
```

Then when it sees:

```text
i like to eat
```

it may predict:

```text
rice
```

because `rice` appeared as a possible next word in the training examples.

In [ ]:
training_text = [
    "i like to eat rice",
    "i like to eat noodles",
    "i like to eat fruit",
]

context = "i like to eat"

print("Context:", context)
print("Possible next words from the training text:")

for sentence in training_text:
    if sentence.startswith(context):
        next_word = sentence.split()[len(context.split())]
        print("-", next_word)

## 3. Tokenization

**Tokenization** means splitting text into smaller pieces called **tokens**.

For this beginner lesson:

```text
one word = one token
```

Example:

```text
I like to eat rice
```

becomes:

```python
["i", "like", "to", "eat", "rice"]
```

We use lowercase so `Rice` and `rice` are treated as the same word.

In [ ]:
sentence = "I like to eat rice"

tokens = sentence.lower().split()

print("Original sentence:")
print(sentence)

print("\nTokens:")
print(tokens)

The code uses:

```python
lower()
```

to make the text lowercase, and:

```python
split()
```

to split the sentence by spaces.

This tokenizer is simple. Real language models usually use more advanced tokenization, but this is enough for our first model.

## 4. Vocabulary

A **vocabulary** is the set of unique tokens that the model knows.

If a word is not in the vocabulary, the simple model cannot represent it.

Example corpus:

```text
i like to eat rice
i like to eat noodles
you like to drink water
```

The vocabulary contains each unique word only once.

In [ ]:
corpus = """
i like to eat rice
i like to eat noodles
you like to drink water
"""

corpus_tokens = corpus.lower().split()
vocabulary = sorted(set(corpus_tokens))

print("Corpus tokens:")
print(corpus_tokens)

print("\nVocabulary:")
print(vocabulary)

print("\nVocabulary size:", len(vocabulary))

Why use `sorted()`?

Because `set()` gives unique words, but the order may not be stable.

Sorting gives a stable vocabulary order, which makes the results reproducible.

## 5. Word IDs

Machine learning models need numbers.

Humans can read words directly:

```text
rice
water
noodles
```

But most machine learning algorithms work with numerical data:

```text
0, 1, 2, 3, ...
```

So we assign each word an ID.

Important:

The ID is only a label. It does not mean one word is bigger, better, or more important than another word.

In [ ]:
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}

print("word_to_id:")
print(word_to_id)

print("\nid_to_word:")
print(id_to_word)

In [ ]:
example_sentence = "i like to eat rice"
example_tokens = example_sentence.split()

word_ids = [word_to_id[word] for word in example_tokens]
converted_back = [id_to_word[word_id] for word_id in word_ids]

print("Sentence:", example_sentence)
print("Tokens:", example_tokens)
print("Word IDs:", word_ids)
print("Converted back:", converted_back)

### Why does machine learning need Word IDs?

Because the model cannot calculate with raw text.

For example, this does not make mathematical sense:

```text
"rice" + "water"
```

But a machine learning model can work with numbers, arrays, vectors, and probabilities.

Word IDs are the first step toward converting text into numerical features.

## 6. N-gram Concept

An **N-gram** is a sequence of N tokens.

Examples:

| N | Name | Example |
|---:|---|---|
| 1 | unigram | `rice` |
| 2 | bigram | `eat rice` |
| 3 | trigram | `to eat rice` |

N-grams help us look at local word patterns.

In [ ]:
def make_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tokens[i:i + n]
        ngrams.append(ngram)
    return ngrams


tokens = "i like to eat rice".split()

print("Tokens:", tokens)
print("\nUnigrams:")
print(make_ngrams(tokens, 1))

print("\nBigrams:")
print(make_ngrams(tokens, 2))

print("\nTrigrams:")
print(make_ngrams(tokens, 3))

## 7. Context Window

A **context window** is the number of previous words used to predict the next word.

If:

```python
context_size = 2
```

then the model uses 2 previous words.

Example:

```text
i like -> to
like to -> eat
to eat -> rice
```

Changing the context size changes what information the model can see.

In [ ]:
tokens = "i like to eat rice".split()

for context_size in [1, 2, 3]:
    print("Context size:", context_size)

    for i in range(len(tokens) - context_size):
        context = tokens[i:i + context_size]
        next_word = tokens[i + context_size]
        print(context, "->", next_word)

    print()

## 8. Context-next Word Pairs

Context-next word pairs are the examples used for training.

Each pair has:

- input: context words
- output: next word

Example:

```text
["i", "like"] -> "to"
```

Later, we will convert the input and output into numbers so a model can learn from them.

In [ ]:
tokens = "i like to eat rice".split()
context_size = 2

pairs = []

for i in range(len(tokens) - context_size):
    context = tokens[i:i + context_size]
    next_word = tokens[i + context_size]
    pairs.append((context, next_word))

print("Context-next word pairs:")
for context, next_word in pairs:
    print(context, "->", next_word)

## 9. Next-word Prediction as Classification

Classification means choosing one class from a set of possible classes.

For next-word prediction:

- input: context words
- possible classes: words in the vocabulary
- correct class: the actual next word

Example:

```text
input: ["i", "like"]
correct output class: "to"
```

If the vocabulary is:

```python
["drink", "eat", "i", "like", "noodles", "rice", "to", "water", "you"]
```

then the model chooses one of these words as the predicted class.

In [ ]:
print("Vocabulary classes:")
for word, class_id in word_to_id.items():
    print("class", class_id, "=", word)

print("\nContext-next pairs with class IDs:")
for context, next_word in pairs:
    class_id = word_to_id[next_word]
    print(context, "->", next_word, "-> class ID:", class_id)

## 10. Basic Feedforward Neural Network Idea

A **Feedforward Neural Network (FNN)** is a neural network where information moves forward:

```text
input layer -> hidden layer -> output layer
```

For our simple language model:

```text
context words -> numerical features -> FNN -> next-word class
```

Example:

```text
["i", "like"] -> FNN -> "to"
```

The FNN needs a fixed-size input.

That is why we use a fixed context window such as 2 words.

If the context size is 2 and the vocabulary size is 9, the output layer has 9 possible classes, one for each vocabulary word.

### FNN Layer Diagram

Feedforward Neural Network layer diagram

In this diagram:

- `x1`, `x2`, `x3` are input features from the context words.
- `h1`, `h2`, `h3`, `h4` are hidden-layer neurons.
- `y1`, `y2`, `y3` are output scores for possible next-word classes.

The arrows show that information moves forward from input to hidden layer to output layer.

In [ ]:
context_size = 2
vocab_size = len(vocabulary)
number_of_output_classes = vocab_size

print("Context size:", context_size)
print("Vocabulary size:", vocab_size)
print("Number of output classes:", number_of_output_classes)

print("\nFNN concept:")
print("context words -> numerical features -> hidden layer -> output word class")

In the next session, we will prepare the numerical features.

Then later, we can use:

```python
from sklearn.neural_network import MLPClassifier
```

`MLPClassifier` is a simple Feedforward Neural Network implementation from scikit-learn.

## 11. Limitation of FNN Language Models

A simple FNN language model is useful for learning, but it has limitations.

### 1. Fixed context window

If the context size is 2, the model only sees 2 previous words.

It cannot naturally use a long paragraph of context.

### 2. Weak memory

An FNN does not naturally remember earlier words from a long sequence.

### 3. Sensitive to small data

If a phrase does not appear in the training data, the model may fail or give a weak prediction.

### 4. Repetition

Generated text may become repetitive because the model only follows short local patterns.

### 5. Motivation for advanced models

RNNs and Transformers are designed to handle sequence and context better than a basic FNN.

For this beginner workflow, the FNN is still useful because it shows the full pipeline clearly:

```text
text -> tokens -> vocabulary -> numerical features -> model -> prediction
```

## Session 1 Summary

You should now understand:

- A language model learns patterns in text.
- Next-word prediction means predicting the next word from previous words.
- Tokenization splits text into tokens.
- Vocabulary stores unique known words.
- Word IDs convert words into numerical labels.
- N-grams are sequences of N tokens.
- Context window controls how many previous words are used.
- Context-next word pairs become training examples.
- Next-word prediction can be classification.
- An FNN maps fixed-size input features to output classes.
- FNN language models have limited context and weak sequence memory.